In [ ]:
import os, shutil, datetime, itertools, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import pandas as pd
from pathlib import Path
from google.colab import drive

In [ ]:
# ---- install keras-tuner if needed ----
try:
    import keras_tuner as kt
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "keras-tuner"])
    import keras_tuner as kt


In [ ]:
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_PT_Aug4000_Results"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)

In [ ]:
IMG_SIZE_RAW = (48, 48)    # grayscale
IMG_SIZE_INP = (224, 224)  # model input size
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augment")

def make_loader(dir_path, shuffle=True, batch=BATCH, augment=False):
    ds = image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="categorical",
        color_mode="grayscale",
        image_size=IMG_SIZE_RAW,
        batch_size=batch,
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_INP)
        x = tf.cast(x, tf.float32) / 255.0  # keep [0,1] → branch-wise preprocess below
        if augment:
            x = aug(x, training=True)
        return x, y

    return ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), class_names

train_ds, class_names = make_loader(LOCAL_TRAIN, shuffle=True,  augment=True)
val_ds, _             = make_loader(LOCAL_VAL,   shuffle=False, augment=False)
test_ds, _            = make_loader(LOCAL_TEST,  shuffle=False, augment=False)
num_classes = len(class_names)
print("Classes:", class_names, "| Num classes:", num_classes)


Found 28000 files belonging to 7 classes.
Found 8616 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'] | Num classes: 7


In [ ]:
# ---------------- HyperModel: Hybrid build ----------------
from tensorflow.keras.applications.resnet import preprocess_input as resnet_pp
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mnetv3_pp
from tensorflow.keras.applications import ResNet101, MobileNetV3Large

def build_hybrid_model(hp: kt.HyperParameters, num_classes: int):
    # small, sensible search
    dense_units = hp.Choice("dense_units", [256, 512], default=512)
    dropout     = hp.Choice("dropout", [0.3, 0.5], default=0.5)
    lr          = hp.Choice("lr", [5e-5, 1e-4, 2e-4], default=1e-4)

    do_ft       = True
    if do_ft:
        unfreeze_last = hp.Choice("unfreeze_last", [20, 30], default=20)
    else:
        unfreeze_last = 0

    # Backbones
    resnet_base = ResNet101(include_top=False, weights="imagenet", input_shape=(224,224,3))
    mnet_base   = MobileNetV3Large(include_top=False, weights="imagenet", input_shape=(224,224,3))
    # freeze all first
    for l in resnet_base.layers: l.trainable = False
    for l in mnet_base.layers:   l.trainable = False

    # optional light finetune (unfreeze last N, keep BN frozen)
    if do_ft and unfreeze_last > 0:
        for l in resnet_base.layers[-unfreeze_last:]:
            if not isinstance(l, layers.BatchNormalization):
                l.trainable = True
        for l in mnet_base.layers[-unfreeze_last:]:
            if not isinstance(l, layers.BatchNormalization):
                l.trainable = True

    inp = layers.Input(shape=(224,224,3), name="inp_rgb01")  # [0,1] RGB

    # Branch-1: ResNet101
    x_r = layers.Lambda(lambda z: resnet_pp(z*255.0), name="resnet_preproc")(inp)
    f_r = resnet_base(x_r, training=False)
    f_r = layers.GlobalAveragePooling2D(name="gap_resnet101")(f_r)

    # Branch-2: MobileNetV3
    x_m = layers.Lambda(lambda z: mnetv3_pp(z*255.0), name="mnetv3_preproc")(inp)
    f_m = mnet_base(x_m, training=False)
    f_m = layers.GlobalAveragePooling2D(name="gap_mobilenetv3")(f_m)

    # Fusion + head
    f = layers.Concatenate(name="concat_feats")([f_r, f_m])
    f = layers.BatchNormalization()(f)
    f = layers.Dense(dense_units, activation="relu")(f)
    f = layers.BatchNormalization()(f)
    f = layers.Dropout(dropout)(f)
    out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

    model = models.Model(inp, out, name="Hybrid_ResNet101_MNetV3")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [ ]:
# ---------------- Tuner ----------------
tuner = kt.RandomSearch(
    hypermodel=lambda hp: build_hybrid_model(hp, num_classes),
    objective="val_accuracy",
    max_trials=2,
    executions_per_trial=1,
    directory=os.path.join(OUT_DIR, "tuner_dir"),
    project_name="hybrid_res101_mnetv3",
    overwrite=True,
    seed=SEED
)

171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
es_cb  = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1)
rlr_cb = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

print("Running tuner.search() with 2 trials...")
tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[es_cb, rlr_cb],
    verbose=1
)

best_hp    = tuner.get_best_hyperparameters(1)[0]
best_model = tuner.get_best_models(1)[0]
print("Best HPs:", best_hp.values)
best_model.save(os.path.join(OUT_DIR, "best_model_tuned3.keras"))


Trial 2 Complete [00h 59m 16s]
val_accuracy: 0.636838436126709

Best val_accuracy So Far: 0.6411327719688416
Total elapsed time: 01h 59m 35s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 58 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Best HPs: {'dense_units': 512, 'dropout': 0.3, 'lr': 0.0001, 'unfreeze_last': 20}


In [ ]:
# ---------------- Short post-train (to log curves) ----------------
def rebuild_from_hp(hp):
    return build_hybrid_model(hp, num_classes)

post_model = rebuild_from_hp(best_hp)
post_model.set_weights(best_model.get_weights())

hpv = best_hp.values
do_ft = hpv.get("do_finetune", False)
unfreeze_last = hpv.get("unfreeze_last", 0)
lr_tuned = hpv.get("lr", 1e-4)
# if finetuning active, step down LR
lr_ft = 1e-5 if (do_ft and unfreeze_last>0) else lr_tuned

post_model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_ft),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

ckpt_path = f"{OUT_DIR}/best_post_model.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    es_cb, rlr_cb
]

ft_history = post_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/8
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step - accuracy: 0.7938 - loss: 0.5698
Epoch 1: val_accuracy improved from -inf to 0.63347, saving model to /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_PT_Aug4000_Results/best_post_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 396s 414ms/step - accuracy: 0.7938 - loss: 0.5698 - val_accuracy: 0.6335 - val_loss: 1.1692 - learning_rate: 1.0000e-04
Epoch 2/8
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step - accuracy: 0.8007 - loss: 0.5393
Epoch 2: val_accuracy did not improve from 0.63347
875/875 ━━━━━━━━━━━━━━━━━━━━ 339s 387ms/step - accuracy: 0.8007 - loss: 0.5393 - val_accuracy: 0.6114 - val_loss: 1.1903 - learning_rate: 1.0000e-04
Epoch 3/8
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.8257 - loss: 0.4895
Epoch 3: val_accuracy did not improve from 0.63347

Epoch 3: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
875/875 ━━━━━━━━━━━━━━━━━━━━ 335s 382ms/step - accuracy: 0.8257 - l

In [ ]:
# ---------------- Curves ----------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(ft_history.history["accuracy"], label="Train Acc")
plt.plot(ft_history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy Curve"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1,2,2)
plt.plot(ft_history.history["loss"], label="Train Loss")
plt.plot(ft_history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
plt.close()
pd.DataFrame(ft_history.history).to_csv(os.path.join(OUT_DIR, "post_history.csv"), index=False)
print("Training curves saved.")


Training curves saved.


In [ ]:
# ---------------- Evaluate on Test ----------------
test_loss, test_acc = post_model.evaluate(test_ds, verbose=0)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")

TEST -> loss: 1.2086 | acc: 0.6225


In [ ]:
# ---------------- Predictions & Reports ----------------
# collect true labels
y_true = [y.numpy() for _, y in test_ds.unbatch()]
y_true = np.array(y_true)
y_true_cls = np.argmax(y_true, axis=1)

# probs & preds
y_prob = post_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Classification Report
rep = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
print(rep)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(rep)

              precision    recall  f1-score   support

       angry     0.5808    0.4875    0.5301       958
   disgusted     0.6374    0.5225    0.5743       111
     fearful     0.5159    0.3955    0.4478      1024
       happy     0.8110    0.8343    0.8225      1774
     neutral     0.5578    0.6139    0.5846      1233
         sad     0.4641    0.5910    0.5199      1247
   surprised     0.7747    0.6787    0.7235       831

    accuracy                         0.6225      7178
   macro avg     0.6203    0.5891    0.6004      7178
weighted avg     0.6275    0.6225    0.6213      7178



In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix (Hybrid tuned3)"); plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=45, ha="right")
plt.yticks(ticks, class_names)
th = cm.max()/2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > th else "black", fontsize=8)
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
print("Confusion matrix saved.")

# ---------------- ROC Curves (OvR + micro) ----------------
y_true_bin = y_true
num_classes = y_true_bin.shape[1]
fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8,6))
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average ROC (AUC = {roc_auc['micro']:.3f})", linewidth=2)
colors = ['aqua','darkorange','cornflowerblue','green','red','purple','brown']
for i, cname in enumerate(class_names):
    plt.plot(fpr[i], tpr[i], lw=2, color=colors[i % len(colors)], label=f"{cname} (AUC = {roc_auc[i]:.3f})")
plt.plot([0,1],[0,1],"k--", lw=1)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC (Hybrid tuned3)")
plt.legend(loc="lower right", fontsize=8)
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"), dpi=150)
plt.close()
print("ROC curves saved:", os.path.join(OUT_DIR, "roc_curves.png"))

print("All outputs saved to:", OUT_DIR)

Confusion matrix saved.
ROC curves saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_PT_Aug4000_Results/roc_curves.png
All outputs saved to: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_ResNet101_MobileNetV3_Hybrid_70_30_PT_Aug4000_Results
